# Domain Adaptation with MMD: MNIST → USPS

This notebook trains a simple CNN classifier on **MNIST** (source) and evaluates on **USPS** (target).
We then **add an MMD loss** to align feature distributions and show improved target performance.

In [1]:
import os, math, random
from typing import Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from PIL import Image

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

device(type='cuda')

## 1) Data: MNIST (source) and USPS (target)

We use torchvision datasets. USPS images are resized to 28×28 and both datasets are scaled to [0,1].

In [2]:
mnist_tf = transforms.ToTensor()
usps_tf  = transforms.Compose([transforms.Resize((28,28)), transforms.ToTensor()])

data_root = "./data"
mnist_train = datasets.MNIST(root=data_root, train=True,  transform=mnist_tf, download=True)
mnist_test  = datasets.MNIST(root=data_root, train=False, transform=mnist_tf, download=True)

# --- USPS with fallback ---
def load_usps_with_fallback(root, train, transform):
    # 1) try torchvision downloader
    try:
        ds = datasets.USPS(root=root, train=train, transform=transform, download=True)
        _ = ds[0]  # touch to ensure files present
        return ds
    except Exception as e:
        print(f"[USPS] torchvision download failed: {e}")

    # 2) fallback to OpenML (needs internet, but different host)
    try:
        from sklearn.datasets import fetch_openml
        print("[USPS] Trying OpenML fallback…")
        openml = fetch_openml("usps", version=2, as_frame=False, data_home=root)
        X = openml["data"]  # shape (9298, 256), values in [-1,1]
        y = openml["target"].astype(np.int64)

        # Split like torchvision’s train/test sizes (7291 train / 2007 test)
        n_train = 7291
        if train:
            X, y = X[:n_train], y[:n_train]
        else:
            X, y = X[n_train:], y[n_train:]

        class USPSOpenML(Dataset):
            def __init__(self, X, y, transform):
                self.X = X.astype(np.float32).reshape(-1, 16, 16)  # [-1,1]
                self.y = y
                self.transform = transform
            def __len__(self):
                return len(self.y)
            def __getitem__(self, idx):
                img = self.X[idx]
                # map from [-1,1] -> [0,1] for display parity with MNIST
                img = (img + 1.0) / 2.0
                img = Image.fromarray((img * 255).astype(np.uint8), mode="L")
                if self.transform is not None:
                    img = self.transform(img)
                return img, int(self.y[idx])

        return USPSOpenML(X, y, transform)
    except Exception as e:
        print(f"[USPS] OpenML fallback also failed: {e}")
        raise RuntimeError(
            "USPS could not be downloaded. If you are offline, manually place USPS files "
            "under ./data/USPS/ and set download=False, or run with internet access."
        )

usps_train = load_usps_with_fallback(data_root, train=True,  transform=usps_tf)
usps_test  = load_usps_with_fallback(data_root, train=False, transform=usps_tf)

len(mnist_train), len(mnist_test), len(usps_train), len(usps_test)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.55MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 129kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.0MB/s]
100%|██████████| 6.58M/6.58M [00:01<00:00, 6.42MB/s]
100%|██████████| 1.83M/1.83M [00:00<00:00, 5.50MB/s]


(60000, 10000, 7291, 2007)

## 2) Model: feature extractor + classifier

In [3]:
class FeatCNN(nn.Module):
    def __init__(self, feat_dim: int = 128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2),  # -> 28x28
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                              # -> 14x14

            nn.Conv2d(32, 64, kernel_size=5, padding=2), # -> 14x14
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                              # -> 7x7
        )
        self.proj = nn.Sequential(
            nn.Flatten(),                                 # 64*7*7
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, feat_dim)
        )
    def forward(self, x):
        h = self.conv(x)
        z = self.proj(h)
        z = F.normalize(z, dim=1)
        return z

class Classifier(nn.Module):
    def __init__(self, feat_dim: int = 128, num_classes: int = 10):
        super().__init__()
        self.fc = nn.Linear(feat_dim, num_classes)
    def forward(self, z):
         return self.fc(z)

class Net(nn.Module):
    def __init__(self, feat_dim: int = 128, num_classes: int = 10):
        super().__init__()
        self.feat = FeatCNN(feat_dim)
        self.cls  = Classifier(feat_dim, num_classes)
    def forward(self, x):
        z = self.feat(x)
        y = self.cls(z)
        return y, z

In [4]:
# Instantiate model
net = Net().to(device)

## 3) Dataloaders

In [5]:
BATCH = 128

num_workers = 0
pin_memory = False
mnist_train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True,
                                drop_last=True, num_workers=num_workers, pin_memory=pin_memory)
mnist_test_loader  = DataLoader(mnist_test,  batch_size=256, shuffle=False,
                                num_workers=num_workers, pin_memory=pin_memory)
usps_train_loader  = DataLoader(usps_train,  batch_size=64, shuffle=True,
                                drop_last=True, num_workers=num_workers, pin_memory=pin_memory)
usps_test_loader   = DataLoader(usps_test,   batch_size=256, shuffle=False,
                                num_workers=num_workers, pin_memory=pin_memory)

# mnist_train_loader = DataLoader(mnist_train, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
# mnist_test_loader  = DataLoader(mnist_test,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
# usps_train_loader  = DataLoader(usps_train,  batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
# usps_test_loader   = DataLoader(usps_test,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

## 4) Evaluation utilities

> Question: Code an evaluation mmethod based on predicting the correct digits from a given image

In [6]:
@torch.no_grad()
def evaluate(model: Net, loader: DataLoader, desc: str = "Eval"):
    model.eval()
    total, correct = 0, 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits, _ = model(x)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
    acc = correct / total
    # model.train()
    return acc

## 5) Baseline: train on MNIST only

In [ ]:
def train_ce_on_mnist(epochs=5, lr=1e-3):
    model = Net().to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for ep in range(1, epochs + 1):
        running_loss, seen, correct = 0.0, 0, 0
        for x, y in tqdm(mnist_train_loader, desc=f"[MNIST CE] Epoch {ep}/{epochs}"):
            x = x.to(device)
            y = y.to(device)
            logits, _ = model(x)
            loss = F.cross_entropy(logits, y)
            optim.zero_grad()
            loss.backward()
            optim.step()
            running_loss += loss.item() * y.size(0)
            seen += y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
        train_loss = running_loss / seen
        train_acc  = correct / seen
        print(f"Epoch {ep}: train_loss={train_loss:.4f}  train_acc={train_acc:.4f}")
    evaluate(model, mnist_test_loader, desc="MNIST→MNIST")
    evaluate(model, usps_test_loader,  desc="MNIST→USPS")
    return model

baseline = train_ce_on_mnist(epochs=10, lr=1e-3)

[MNIST CE] Epoch 1/10:   0%|          | 0/937 [00:00<?, ?it/s]

Epoch 1: train_loss=0.4419  train_acc=0.9626


[MNIST CE] Epoch 2/10:   0%|          | 0/937 [00:00<?, ?it/s]

Epoch 2: train_loss=0.0739  train_acc=0.9846


[MNIST CE] Epoch 3/10:   0%|          | 0/937 [00:00<?, ?it/s]

## 6) MMD: RBF kernel and unbiased estimator

In [ ]:
# Code a mixture of RBF Kernels with different bandwidths
def rbf_kernel(x: torch.Tensor, y: torch.Tensor, sigmas=(2, 5, 10)) -> torch.Tensor:
    sq = (x[:, None, :] - y[None, :, :]).pow(2).sum(dim=2)
    k = 0.0
    for s in sigmas:
        k = k + torch.exp(-sq / (2.0 * (float(s) ** 2)))
    return k

#Code the unbiased MMD estimate
def mmd_unbiased(x: torch.Tensor, y: torch.Tensor, sigmas=(2,5,10)) -> torch.Tensor:
    n, m = x.size(0), y.size(0)
    Kxx = rbf_kernel(x, x, sigmas)
    Kyy = rbf_kernel(y, y, sigmas)
    Kxy = rbf_kernel(x, y, sigmas)
    mask_x = ~torch.eye(n, dtype=torch.bool, device=x.device)
    mask_y = ~torch.eye(m, dtype=torch.bool, device=y.device)
    term_xx = Kxx[mask_x].mean() if n > 1 else Kxx.new_zeros(())
    term_yy = Kyy[mask_y].mean() if m > 1 else Kyy.new_zeros(())
    term_xy = Kxy.mean()
    mmd2 = term_xx + term_yy - 2.0 * term_xy
    return mmd2

> Question: Compare the unbiased and biased estimator MMD. Hint: Plot the matrix with the same scale

In [ ]:
def mmd_biased(x, y, sigmas=(2.,5.,10.)):
    Kxx = rbf_kernel(x, x, sigmas).mean()
    Kyy = rbf_kernel(y, y, sigmas).mean()
    Kxy = rbf_kernel(x, y, sigmas).mean()
    return (Kxx + Kyy - 2*Kxy).clamp_min(0.0)

def plot_kernels_same_scale(x, y, sigmas=(2.,5.,10.), title_prefix=""):
    Kxx = rbf_kernel(x, x, sigmas).cpu()
    Kyy = rbf_kernel(y, y, sigmas).cpu()
    Kxy = rbf_kernel(x, y, sigmas).cpu()
    vmin = min(Kxx.min().item(), Kyy.min().item(), Kxy.min().item())
    vmax = max(Kxx.max().item(), Kyy.max().item(), Kxy.max().item())
    fig, axs = plt.subplots(1, 3, figsize=(12, 3))
    ims = []
    ims.append(axs[0].imshow(Kxx, vmin=vmin, vmax=vmax))
    axs[0].set_title(f"{title_prefix}Kxx")
    ims.append(axs[1].imshow(Kyy, vmin=vmin, vmax=vmax))
    axs[1].set_title(f"{title_prefix}Kyy")
    ims.append(axs[2].imshow(Kxy, vmin=vmin, vmax=vmax))
    axs[2].set_title(f"{title_prefix}Kxy")
    for ax in axs: ax.axis("off")
    cbar = fig.colorbar(ims[0], ax=axs, fraction=0.02, pad=0.04)
    cbar.set_label("kernel value")
    plt.show()

x_s, _ = next(iter(mnist_test_loader))
x_t, _ = next(iter(usps_test_loader))
model = baseline if 'baseline' in globals() else Net().to(device)
model.eval()

with torch.no_grad():
    z_s = model.feat(x_s.to(device))
    z_t = model.feat(x_t.to(device))

N = min(z_s.size(0), z_t.size(0), 128)
z_s = z_s[:N].detach()
z_t = z_t[:N].detach()

plot_kernels_same_scale(z_s, z_t, sigmas=(2.,5.,10.), title_prefix="MNIST↔USPS")

## Answer:

On the one hand, the biased estimator includes the bright diagonal in Kxx/Kyy, yielding a slightly larger and more stable MMD. On the other hand, the unbiased estimator ignores that diagonal and gives a more faithful but noisier MMD (sometimes slightly negative on small batches).

## 7) Domain adaptation with MMD

In [ ]:
def train_mmd_adapt(epochs=10, lr=1e-3, lam=1.0, sigmas=(2,5,10), mmd_fn=mmd_unbiased):
    model = Net().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ce  = nn.CrossEntropyLoss()

    for ep in range(1, epochs + 1):
        model.train()
        running_loss = running_ce = running_mmd = 0.0
        seen = 0
        tgt_iter = iter(usps_train_loader)
        for xs, ys in tqdm(mnist_train_loader, desc=f"[MMD] Epoch {ep}/{epochs} (λ={lam:.2f})"):
            try:
                xt, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(usps_train_loader)
                xt, _ = next(tgt_iter)
            xs, ys = xs.to(device), ys.to(device)
            xt     = xt.to(device)
            logits_s, z_s = model(xs)
            z_t = model.feat(xt)
            loss_ce  = ce(logits_s, ys)
            loss_mmd = mmd_fn(z_s, z_t, sigmas=sigmas)
            loss     = loss_ce + lam * loss_mmd
            opt.zero_grad()
            loss.backward()
            opt.step()
            bsz = ys.size(0)
            seen         += bsz
            running_loss += loss.item()     * bsz
            running_ce   += loss_ce.item()  * bsz
            running_mmd  += loss_mmd.item() * bsz

        print(f"Epoch {ep}: loss={running_loss/seen:.4f} | CE={running_ce/seen:.4f} | MMD={running_mmd/seen:.4f}")
        evaluate(model, mnist_test_loader, desc="Adapted MNIST→MNIST")
        evaluate(model, usps_test_loader,  desc="Adapted MNIST→USPS")
    return model

adapted = train_mmd_adapt(epochs=10, lr=1e-3, lam=1.0, sigmas=(2,5,10))

## 8) Compare baseline vs adapted

In [ ]:
print("Baseline (MNIST-only):")
b_mnist = evaluate(baseline, mnist_test_loader, "MNIST→MNIST")
b_usps = evaluate(baseline, usps_test_loader,  "MNIST→USPS")
print(f"  MNIST test: {b_mnist*100:.2f}% | USPS test: {b_usps*100:.2f}%")

print("\nAdapted (MNIST + MMD to USPS):")
a_mnist = evaluate(adapted, mnist_test_loader, "MNIST→MNIST")
a_usps = evaluate(adapted, usps_test_loader,  "MNIST→USPS")
print(f"  MNIST test: {a_mnist*100:.2f}% | USPS test: {a_usps*100:.2f}%")

> Question: Repeat the experiment with the biased estimator.

In [ ]:
adapted_biased = train_mmd_adapt(epochs=10, lr=1e-3, lam=1.0, sigmas=(2.,5.,10.), mmd_fn=mmd_biased)

In [ ]:
print("\nAdapted (biased MMD):")
bi_mnist = evaluate(adapted_biased, mnist_test_loader, "MNIST→MNIST")
bi_usps  = evaluate(adapted_biased, usps_test_loader,  "MNIST→USPS")
print(f"MNIST: {bi_mnist*100:.2f}% | USPS: {bi_usps*100:.2f}%")

# 9) IPM vs Divergences

> Question: Compare the probability distribution, code a moving gaussian in which the average moves with a parameter $x$, where $x$ is a distance from the mean 0 gaussian (centered gaussian). Plot the MMD in function of $x$ vs KL-divergene in function of $x$. In both cases, the reference distribution is the centered gaussian. What is happening?

In [ ]:
def mmd2_gaussian_closed_form(x, sigma_k=1.0, var=1.0):
    c = sigma_k**2
    a = (c/(c+2*var))**0.5
    return 2*a*(1 - np.exp(-x**2/(2*(c+2*var))))

def kl_same_var(x, var=1.0):
    return 0.5*(x**2)/var

xs = np.linspace(0.0, 3.0, 301)
mmd_vals = [mmd2_gaussian_closed_form(x, sigma_k=1.0, var=1.0) for x in xs]
kl_vals  = [kl_same_var(x, var=1.0) for x in xs]

plt.figure(figsize=(7,4))
plt.plot(xs, mmd_vals, label="MMD² (RBF σk=1, closed form)")
plt.plot(xs, kl_vals,  label="KL(P||Q), P=N(0,1), Q=N(x,1)")
plt.xlabel("mean shift x")
plt.ylabel("value")
plt.title("MMD vs KL as the Gaussian mean moves")
plt.legend()
plt.show()

## Answer:

For small shifts $x$, both KL and MMD$^2$ increase approximately quadratically.

As $x$ becomes large, KL grows without bound ($\tfrac{x^2}{2}$).

MMD$^2$ with an RBF kernel saturates (the kernel no longer “sees” the other Gaussian beyond its bandwidth).

## Bonus: KL vs MMD for generated vs real samples

Compare a **non-pretrained** tiny generator vs real MNIST distribution using KL and MMD.


In [ ]:
class TinyGen(nn.Module):
    """A very small, untrained generator for demonstration purposes.
       zdim: dimension of the input noise vector
    """
    def __init__(self, zdim=64):
        super().__init__()
        self.zdim = zdim
        self.net = nn.Sequential(
            nn.Linear(zdim, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 512),  nn.ReLU(inplace=True),
            nn.Linear(512, 28*28), nn.Sigmoid()
        )
    def forward(self, n=512):
        z = torch.randn(n, self.zdim, device=device)
        x = self.net(z).view(n, 1, 28, 28)
        return x

# -------- Real samples from MNIST (assumes mnist_train exists) --------
@torch.no_grad()
def sample_real_images(n=512):
    idx = torch.randperm(len(mnist_train))[:n]
    xs = torch.stack([mnist_train[i][0] for i in idx], dim=0)  # [n,1,28,28], in [0,1]
    return xs.to(device)

# -------- KL & MMD utilities --------
def kl_discrete(p_hist, q_hist, eps=0.0):
    p = np.asarray(p_hist, dtype=np.float64)
    q = np.asarray(q_hist, dtype=np.float64)
    if eps > 0.0:
        p = p + eps
        q = q + eps
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    if eps == 0.0:
        if np.any((p > 0) & (q == 0)):
            return float("inf")
    mask = p > 0
    return float(np.sum(p[mask] * (np.log(p[mask]) - np.log(q[mask]))))

@torch.no_grad()
def mmd_images(x_real, x_fake, sigmas=(5,10,20)):
    #compute MMD between two image sets
    xr = x_real.view(x_real.size(0), -1)   # [N, 784]
    xf = x_fake.view(x_fake.size(0), -1)
    def rbf_mix(a, b):
        d2 = torch.cdist(a, b, p=2.0).pow(2)  # [N,M]
        k  = 0.
        for s in sigmas:
            k = k + torch.exp(-d2 / (2.0 * (s**2)))
        return k / len(sigmas)
    N = xr.size(0); M = xf.size(0)
    Kxx = rbf_mix(xr, xr)
    Kyy = rbf_mix(xf, xf)
    Kxy = rbf_mix(xr, xf)
    mask_x = ~torch.eye(N, dtype=torch.bool, device=xr.device)
    mask_y = ~torch.eye(M, dtype=torch.bool, device=xf.device)
    term_xx = Kxx[mask_x].mean() if N > 1 else Kxx.new_zeros(())
    term_yy = Kyy[mask_y].mean() if M > 1 else Kyy.new_zeros(())
    term_xy = Kxy.mean()
    mmd2 = (term_xx + term_yy - 2.0 * term_xy).clamp_min(0.0)
    return float(mmd2)

# -------- Run once (non-pretrained) --------
gen_bad = TinyGen().to(device).eval()
real = sample_real_images(512)
fake_bad = gen_bad(512).clamp(0,1)

bins = np.linspace(0, 1, 257)
ph, _ = np.histogram(real.detach().cpu().numpy().ravel(), bins=bins)
qh_bad, _ = np.histogram(fake_bad.detach().cpu().numpy().ravel(), bins=bins)  # <-- detach!

kl_raw_bad = kl_discrete(ph, qh_bad, eps=0.0)
kl_smooth_bad = kl_discrete(ph, qh_bad, eps=1e-8)
mmd_bad = mmd_images(real, fake_bad)

print("Non-pretrained generator vs real:")
print(f"  KL (raw)     : {kl_raw_bad}")
print(f"  KL (smoothed): {kl_smooth_bad:.3f}")
print(f"  MMD          : {mmd_bad:.3f}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# -------- A tiny discriminator (CNN) --------
class TinyDisc(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),  # 14x14
            nn.Conv2d(16, 32, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),  # 7x7
            nn.Flatten(),
            nn.Linear(32*7*7, 64), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(64, 1)  # logits
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

# -------- Instantiate G and D --------
gen = TinyGen().to(device)
disc = TinyDisc().to(device)

optG = torch.optim.Adam(gen.parameters(),  lr=2e-4, betas=(0.5, 0.999))
optD = torch.optim.Adam(disc.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce  = nn.BCEWithLogitsLoss()

# -------- Metrics setup --------
EPOCHS = 60
BATCH  = 256
KL_raw, KL_smooth, MMD_vals, G_losses, D_losses = [], [], [], [], []

# Fix a reference real batch (for consistent KL/MMD across epochs)
with torch.no_grad():
    real_ref = sample_real_images(512)    # [512,1,28,28]
bins = np.linspace(0, 1, 257)
ph_ref, _ = np.histogram(real_ref.detach().cpu().numpy().ravel(), bins=bins)

for epoch in range(1, EPOCHS+1):
    # -----------------------
    # 1) Train Discriminator
    # -----------------------
    disc.train(); gen.train()
    real = sample_real_images(BATCH)
    fake = gen(BATCH).detach()
    logits_real = disc(real)
    logits_fake = disc(fake)
    d_loss = bce(logits_real, torch.ones_like(logits_real)) + \
             bce(logits_fake, torch.zeros_like(logits_fake))
    optD.zero_grad(); d_loss.backward(); optD.step()

    # -----------------------
    # 2) Train Generator
    # -----------------------
    fake = gen(BATCH)
    logits_fake = disc(fake)
    g_loss = bce(logits_fake, torch.ones_like(logits_fake))
    optG.zero_grad(); g_loss.backward(); optG.step()

    # -----------------------
    # 3) Metrics (KL / MMD)
    # -----------------------
    with torch.no_grad():
        fake_eval = gen(512).clamp(0,1)
        qh, _ = np.histogram(fake_eval.detach().cpu().numpy().ravel(), bins=bins)
        KL_raw.append(   kl_discrete(ph_ref, qh, eps=0.0) )
        KL_smooth.append(kl_discrete(ph_ref, qh, eps=1e-8))
        MMD_vals.append( mmd_images(real_ref, fake_eval) )
        G_losses.append(float(g_loss.item()))
        D_losses.append(float(d_loss.item()))

# -----------------------
# 4) Plots
# -----------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# (a) KL
axes[0].plot(range(1, EPOCHS+1), KL_raw, 'o-', label='KL (raw)')
axes[0].plot(range(1, EPOCHS+1), KL_smooth, 'o--', label='KL (smoothed)')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("KL"); axes[0].set_title("KL vs Epoch")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# (b) MMD
axes[1].plot(range(1, EPOCHS+1), MMD_vals, 'o-', color='teal', label='MMD')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MMD"); axes[1].set_title("MMD vs Epoch")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# (c) GAN losses
axes[2].plot(range(1, EPOCHS+1), D_losses, '-', label='D loss')
axes[2].plot(range(1, EPOCHS+1), G_losses, '-', label='G loss')
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Loss"); axes[2].set_title("GAN Losses")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

> Question: What we can conclude about this results in this toy experiment? How does it affect real GAN training?

## Answer:

In this toy setup, both KL and MMD drop as the generator approaches the data, but they behave differently. Raw KL is unforgiving when the generator misses support (can be infinite) and often shows sudden drops once previously empty regions get some mass; smoothed KL makes this visible but finite. MMD provides a smooth, bounded signal that decreases steadily. But it can saturate once distributions separate beyond the kernel scale, making it less sensitive to far-away modes.

For real GANs, this implies we shouldn’t rely on adversarial loss alone. Oscillations are normal and don’t reliably track sample quality.